# Digital DNA — 04: Fusion Experiments & ROC/EER Benchmarking
Multi-modality score fusion, sliding-window temporal smoothing, and Equal Error Rate (EER) evaluation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from fusion.score_fusion import ScoreFusion
from fusion.trust_score import TrustScoreEngine
from fusion.decision_engine import DecisionEngine
from scripts.evaluate_performance import compute_eer

## 1. Multi-Modality Fusion Simulation

In [ ]:
from models.model_utils import load_model

k_model = load_model(str(project_root / "data" / "profiles" / "user_001" / "keystroke_model.joblib"))
m_model = load_model(str(project_root / "data" / "profiles" / "user_001" / "mouse_model.joblib"))

gen_k = pd.read_csv(project_root / "data" / "processed" / "genuine_keystroke.csv").drop(columns=["label"], errors="ignore").values
imp_k = pd.read_csv(project_root / "data" / "processed" / "impostor_keystroke.csv").drop(columns=["label"], errors="ignore").values
gen_m = pd.read_csv(project_root / "data" / "processed" / "genuine_mouse.csv").drop(columns=["label"], errors="ignore").values
imp_m = pd.read_csv(project_root / "data" / "processed" / "impostor_mouse.csv").drop(columns=["label"], errors="ignore").values

fusion = ScoreFusion(weights={"keystroke": 0.5, "mouse": 0.5})

n = min(len(gen_k), len(gen_m))
gen_fused = [fusion.fuse({"keystroke": k_model.score(gen_k[i]), "mouse": m_model.score(gen_m[i])}) for i in range(n)]
imp_fused = [fusion.fuse({"keystroke": k_model.score(imp_k[i]), "mouse": m_model.score(imp_m[i])}) for i in range(n)]

print(f"Fused Genuine Similarity Mean:  {np.mean(gen_fused):.3f}")
print(f"Fused Impostor Similarity Mean: {np.mean(imp_fused):.3f}")

## 2. Compute Equal Error Rate (EER)

In [ ]:
y_true = np.array([1] * len(gen_fused) + [0] * len(imp_fused))
y_scores = np.array(gen_fused + imp_fused)

eer, opt_thresh, fpr, fnr, thresholds = compute_eer(y_true, y_scores)

print(f"Fused Multi-Modality EER: {eer * 100:.2f}%")
print(f"Optimal Operating Threshold: {opt_thresh:.4f}")